# Этап 3, версия 1.7. Проверка диапазона цен 10–90%

Увеличиваем нижний порог с 1-го до 10-го процентиля. Цены уже приведены к TRY, подозрительная запись № 392157 исключена. Сначала отбираем диапазон, затем заново делим объявления по группам на train/validation/test. Обучаем те же LightGBM и CatBoost и сравниваем их на одинаковой validation. Test не используем.

Порог выбирается по известным ценам для исследовательского сравнения. Будущую цену объекта мы заранее не знаем, поэтому этот фильтр не является готовым правилом для приложения.


In [ ]:
# Получаем продажи в TRY и функцию создания признаков из этапа 2.
import json
from pathlib import Path
from IPython.utils.capture import capture_output
stage2 = next((p / "notebooks" / "02_preprocessing_v1.7.ipynb"
               for p in (Path.cwd(), Path.cwd().parent)
               if (p / "notebooks" / "02_preprocessing_v1.7.ipynb").exists()), None)
if stage2 is None:
    raise FileNotFoundError("Не найден этап 2 v1.7")
with capture_output():
    for cell in json.loads(stage2.read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code":
            result = get_ipython().run_cell("".join(cell["source"]))
            if result.error_before_exec or result.error_in_exec:
                raise result.error_before_exec or result.error_in_exec
print("Продаж после исключения проверенной аномалии:", len(sale))


## 1. Границы и новое разбиение

Перцентили считаем по всем положительным ценам продаж после перевода в TRY и исключения записи № 392157. Не смешиваем продажи с арендой и цены в разных валютах. Для точного повторения предложенного фильтра используем строгие неравенства `>` и `<`. Это **исследовательский отбор по целевой цене**, а не способ подготовить модель для прогноза любого объекта.
 Здесь используются 10-й и 90-й процентили цены.


In [ ]:
# Сначала фильтруем цены, затем делим отобранные объявления по группам.
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

q10, q90 = sale.price.quantile([.10, .90])
selected = sale.loc[sale.price.gt(q10) & sale.price.lt(q90)].copy()
group_cols = ["address", "sub_type", "room_count", "size"]
group_selected = pd.util.hash_pandas_object(selected[group_cols], index=False)
first = GroupShuffleSplit(n_splits=1, test_size=.15, random_state=42)
dev_idx, test_idx = next(first.split(selected, groups=group_selected))
dev, range_test = selected.iloc[dev_idx], selected.iloc[test_idx]
second = GroupShuffleSplit(n_splits=1, test_size=.15/.85, random_state=43)
train_idx, val_idx = next(second.split(dev, groups=group_selected.iloc[dev_idx]))
range_train, range_val = dev.iloc[train_idx], dev.iloc[val_idx]

# Полная модель видит и цены вне диапазона, но ни одну группу validation/test.
holdout_groups = set(pd.util.hash_pandas_object(
    pd.concat([range_val, range_test])[group_cols], index=False))
group_all = pd.util.hash_pandas_object(sale[group_cols], index=False)
full_train = sale.loc[~group_all.isin(holdout_groups)].copy()
assert set(range_train.id).issubset(set(full_train.id))
assert set(range_train.id).isdisjoint(set(range_val.id))
assert set(range_train.id).isdisjoint(set(range_test.id))
assert set(range_val.id).isdisjoint(set(range_test.id))
assert set(group_all.loc[full_train.index]).isdisjoint(holdout_groups)
display(pd.DataFrame({
    "Набор": ["Все продажи", "Диапазон 10–90%", "Train диапазона", "Validation диапазона", "Test диапазона", "Train полной модели"],
    "Объявлений": [len(sale), len(selected), len(range_train), len(range_val), len(range_test), len(full_train)]
}))
print(f"Границы: {q10:,.0f} < price < {q90:,.0f} TRY")
print(f"Исключено из диапазона: {len(sale)-len(selected):,} объявлений ({100*(1-len(selected)/len(sale)):.1f}%)")


## 2. Одинаковая подготовка и обучение

Признаки создаются одинаково. Недостающую площадь заполняем по микрорайону и типу жилья, затем по району, типу и общей медиане. Эти медианы, границы площади, кодирование категорий и числовые заполнители вычисляются **отдельно по train каждой модели**. Алгоритм и его настройки совпадают; отличается только набор тренировочных строк.


In [ ]:
# Обучаем две модели; validation и test не участвуют в fit.
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMRegressor

def make_features(train_part, val_part):
    x_train, x_val = features(train_part), features(val_part)
    by_neighborhood = x_train.groupby(["neighborhood", "sub_type"]).size_m2.agg(["median", "count"])
    by_neighborhood = by_neighborhood.loc[by_neighborhood["count"].ge(20), "median"]
    by_district = x_train.groupby(["district", "sub_type"]).size_m2.agg(["median", "count"])
    by_district = by_district.loc[by_district["count"].ge(20), "median"]
    by_type = x_train.groupby("sub_type").size_m2.median()
    overall = x_train.size_m2.median()
    def fill_area(x):
        x = x.copy()
        for keys, medians in [(["neighborhood", "sub_type"], by_neighborhood),
                              (["district", "sub_type"], by_district)]:
            idx = pd.MultiIndex.from_frame(x[keys])
            local = pd.Series(medians.reindex(idx).to_numpy(), index=x.index)
            x["size_m2"] = x.size_m2.fillna(local)
        x["size_m2"] = x.size_m2.fillna(x.sub_type.map(by_type)).fillna(overall)
        return x
    x_train, x_val = fill_area(x_train), fill_area(x_val)
    low, high = x_train.size_m2.quantile([.001, .999])
    for x in (x_train, x_val):
        x["size_m2"] = x.size_m2.clip(low, high)
    return x_train, x_val

def fit_model(train_part):
    x_train, x_val = make_features(train_part, range_val)
    encoder = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", Pipeline([
            ("fill", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50))
        ]), cat)
    ], sparse_threshold=.8)
    reg = TransformedTargetRegressor(
        regressor=LGBMRegressor(random_state=42, verbosity=-1, n_jobs=4, num_leaves=15),
        func=np.log1p, inverse_func=np.expm1)
    model = Pipeline([("encode", encoder), ("model", reg)])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(x_train, train_part.price)
        pred = model.predict(x_val)
    return model, pred

full_model, full_pred = fit_model(full_train)
range_model, range_pred = fit_model(range_train)
print("Обе модели обучены. Validation:", len(range_val), "объявлений; test не использован.")


## 1. Сравниваемые признаки

Площадь и остальные признаки получаем теми же правилами, что в этого диапазона. Добавляем `days_since_start` — число дней от первой даты в train до публикации объявления. Вычисление опорной даты использует только train. Для CatBoost категории передаются как текст; редкие микрорайоны не объединяются вручную. Числовые пропуски заполняются медианой своего train.


In [ ]:
# Создаём одинаковую таблицу признаков для моделей с датой.
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             median_absolute_error, mean_absolute_percentage_error, r2_score)
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings

x_train, x_val = make_features(range_train, range_val)
origin = range_train.listing_date.min()
x_train["days_since_start"] = (range_train.listing_date - origin).dt.days
x_val["days_since_start"] = (range_val.listing_date - origin).dt.days
numeric = num + ["days_since_start"]
assert x_train[numeric].shape[1] == x_val[numeric].shape[1]
print("Период:", range_train.listing_date.min().date(), "—", range_train.listing_date.max().date())
print("Разных микрорайонов train:", x_train.neighborhood.nunique())


## 2. Обучение

LightGBM с датой использует тот же One-Hot-подход и настройки, что модель версии 1.5; меняется только один признак. CatBoost получает категории напрямую. Все модели обучаются только на train. Для честной проверки не подбираем число итераций по этой же validation и не используем test.


In [ ]:
# Вариант LightGBM: проверяем вклад даты при прежнем кодировании.
onehot = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50))
    ]), cat)
], sparse_threshold=.8)
lightgbm_date = Pipeline([
    ("encode", onehot),
    ("model", TransformedTargetRegressor(
        regressor=LGBMRegressor(random_state=42, verbosity=-1, n_jobs=4, num_leaves=15),
        func=np.log1p, inverse_func=np.expm1))
])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lightgbm_date.fit(x_train, range_train.price)
    pred_lightgbm_date = lightgbm_date.predict(x_val)
print("LightGBM с датой обучена")

# Вариант CatBoost: сохраняем детальные текстовые категории.
cat_train, cat_val = x_train[numeric + cat].copy(), x_val[numeric + cat].copy()
for column in cat:
    cat_train[column] = cat_train[column].fillna("Неизвестно").astype(str)
    cat_val[column] = cat_val[column].fillna("Неизвестно").astype(str)
medians = cat_train[numeric].median()
cat_train[numeric] = cat_train[numeric].fillna(medians).astype(float)
cat_val[numeric] = cat_val[numeric].fillna(medians).astype(float)
cat_model = CatBoostRegressor(iterations=600, depth=8, learning_rate=0.08,
                              loss_function="RMSE", random_seed=42,
                              thread_count=4, verbose=False, allow_writing_files=False)
cat_model.fit(cat_train, np.log1p(range_train.price), cat_features=cat)
pred_catboost = np.expm1(cat_model.predict(cat_val))
print("CatBoost обучена; test не использовался")


## 3. Метрики на общей validation

Сначала оцениваем все типы жилья в выбранном диапазоне. Затем на тех же предсказаниях отдельно рассматриваем квартиры с исходно известной площадью и без неё. Заполненная медианой площадь не считается известной. MAE, RMSE и MedAE — в TRY, MAPE — в процентах.


In [ ]:
# Одинаковые объекты и одна шкала метрик для всех трёх моделей.
predictions = {
    "LightGBM без даты": range_pred,
    "LightGBM + дата": pred_lightgbm_date,
    "CatBoost + дата": pred_catboost,
}
area = pd.to_numeric(range_val["size"], errors="coerce")
groups = {
    "Все объекты": np.ones(len(range_val), dtype=bool),
    "Квартиры, площадь известна": (range_val.sub_type.eq("Daire") & area.ge(15)).to_numpy(),
    "Квартиры, площадь неизвестна": (range_val.sub_type.eq("Daire") & ~area.ge(15)).to_numpy(),
}
rows = []
y = range_val.price.to_numpy()
for group_name, mask in groups.items():
    for model_name, pred in predictions.items():
        rows.append({
            "Группа": group_name, "Модель": model_name, "Объектов": int(mask.sum()),
            "MAE, TRY": mean_absolute_error(y[mask], pred[mask]),
            "RMSE, TRY": np.sqrt(mean_squared_error(y[mask], pred[mask])),
            "MedAE, TRY": median_absolute_error(y[mask], pred[mask]),
            "MAPE, %": 100*mean_absolute_percentage_error(y[mask], pred[mask]),
            "R²": r2_score(y[mask], pred[mask]),
        })
comparison = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
for group_name in groups:
    print(group_name)
    display(comparison.loc[comparison["Группа"].eq(group_name)].drop(columns="Группа").round(3))


## 4. Интерпретация

После ограничения цены диапазоном **138 000–750 000 TRY** осталось **228 032 объявления** — 79,6% сегмента продаж. Из него 159 531 попало в train, 34 620 — в validation, 33 881 — в нетронутый test.

На одинаковых 34 620 объявлениях validation CatBoost получил **MAE 55 804 TRY, MAPE 17,98%, R² 0,662**. LightGBM получил **MAE 67 505 TRY, MAPE 21,92%, R² 0,528**. Добавление даты к LightGBM метрики не изменило. На квартирах с известной площадью R² CatBoost составил **0,672**, с отсутствующей — **0,612**.

В версии 1.6 (диапазон 1–90%) у CatBoost были MAE **52 500 TRY** и R² **0,711**. Численно новый порог **не улучшил** эти результаты, при этом исключил ещё 26 270 объявлений. Однако сравнение двух версий не является парным: изменились состав validation и разброс цен. Для окончательного выбора порога нужен эксперимент на фиксированной общей проверочной выборке.

Результаты относятся к сегменту, выбранному по известной целевой цене. При оценке нового объявления цена заранее неизвестна, поэтому эти метрики нельзя переносить на все продажи. Финальный test не использован.
